In [1]:
import sys
import os
import numpy as np
import yaml
import logging
from pathlib import Path
import time

import matplotlib.pyplot as plt
%matplotlib inline

# --- Add paths for custom modules ---
# Adjust these paths if your notebook is in a different location
# Option 1: If notebook is in repo_root/notebooks and helpers in repo_root/notebooks
# Option 2: If notebook is in repo_root/test_py and helpers in repo_root/test_py
# Option 3: If notebook is elsewhere, point to the correct locations

# Assuming the notebook is in a 'notebooks' subdirectory of the repository root
try:
    repo_root = Path.cwd().parent
    build_dir = repo_root / 'build' # For mpy_detector.so
    helpers_dir = Path.cwd() # Assuming notebook_helpers.py is in the same dir as the notebook

    if str(build_dir) not in sys.path:
        sys.path.insert(0, str(build_dir))
    if str(helpers_dir) not in sys.path:
        sys.path.insert(0, str(helpers_dir))
    log_path_added = True
except Exception as e:
    print(f"Error setting up sys.path: {e}")
    log_path_added = False

# --- Import custom and NuScenes modules ---
try:
    import mpy_detector as mdet
    if not log_path_added: # Attempt to import helpers if path setup failed but they are findable
        import notebook_helpers as nh
    else: # Normal import after path setup
        import notebook_helpers as nh

    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    from pyquaternion import Quaternion
except ImportError as e:
    print(f"CRITICAL: Failed to import mpy_detector, notebook_helpers, or NuScenes components: {e}")
    print(f"Ensure mpy_detector.so is in the build directory ({build_dir if 'build_dir' in locals() else 'UNKNOWN'}) and it's in sys.path.")
    print(f"Ensure notebook_helpers.py is in {helpers_dir if 'helpers_dir' in locals() else 'UNKNOWN'} and it's in sys.path.")
    print(f"Python sys.path: {sys.path}")
    # You might want to raise an error here or skip parts of the notebook
    mdet = None
    nh = None


# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
log = logging.getLogger("NuScenesNotebook")
if nh: log.info("Notebook helpers imported.")
if mdet: log.info("mpy_detector imported.")
log.info("NuScenes devkit available.")

# --- Configuration ---
# Assuming notebook is in a subdir of repo_root
CONFIG_PATH_STR = str(repo_root / "test_data/configs/test_full_config_low_log.yaml")
if not Path(CONFIG_PATH_STR).exists():
    log.error(f"Config file not found: {CONFIG_PATH_STR}")
    CONFIG_PATH_STR = None # Handle this downstream

NU_DATAROOT = None
NU_VERSION = 'v1.0-mini' # or 'v1.0-trainval'
if CONFIG_PATH_STR:
    try:
        with open(CONFIG_PATH_STR, 'r') as f:
            full_config = yaml.safe_load(f)
        if full_config and 'dataset' in full_config:
            NU_DATAROOT_RAW = full_config['dataset'].get('dataroot')
            if NU_DATAROOT_RAW:
                NU_DATAROOT = str(Path(NU_DATAROOT_RAW).expanduser())
            NU_VERSION = full_config['dataset'].get('version', NU_VERSION)
        log.info(f"Using NuScenes dataroot: {NU_DATAROOT}, version: {NU_VERSION}")
    except Exception as e:
        log.error(f"Error loading dataset config from {CONFIG_PATH_STR}: {e}")
else:
    log.warning("Config path not set. NuScenes dataroot and version might be incorrect.")

# --- Parameters for this run ---
NU_SCENE_INDEX_TO_LOAD = 0  # Which scene to load from NuScenes
NU_NUM_SAMPLES_TO_PROCESS = 10 # How many consecutive lidar scans to process from the scene
# For visualization
PLOT_X_LIMS = (-50, 50) # Bird's eye view X limits
PLOT_Y_LIMS = (-50, 50) # Bird's eye view Y limits
POINT_SIZE = 2

2025-05-07 17:02:24,153 - NuScenesNotebook - INFO - Notebook helpers imported.
2025-05-07 17:02:24,154 - NuScenesNotebook - INFO - mpy_detector imported.
2025-05-07 17:02:24,155 - NuScenesNotebook - INFO - NuScenes devkit available.
2025-05-07 17:02:24,169 - NuScenesNotebook - INFO - Using NuScenes dataroot: /datasets/nuscenes, version: v1.0-mini


In [2]:
if mdet is None or CONFIG_PATH_STR is None:
    log.error("mpy_detector or config path not available. Cannot initialize filter.")
    filt = None
else:
    log.info(f"Initializing DynObjFilter with config: {CONFIG_PATH_STR}")
    try:
        filt = mdet.DynObjFilter(config_path=CONFIG_PATH_STR)
        log.info("DynObjFilter initialized successfully.")
        log.info(f"Filter initial map count: {filt.get_depth_map_count()}")
    except Exception as e:
        log.error(f"Error initializing DynObjFilter: {e}", exc_info=True)
        filt = None

nusc = None
if NU_DATAROOT and Path(NU_DATAROOT).exists():
    log.info(f"Loading NuScenes (Version: {NU_VERSION}, Dataroot: {NU_DATAROOT})...")
    try:
        nusc = NuScenes(version=NU_VERSION, dataroot=NU_DATAROOT, verbose=False)
        log.info("NuScenes loaded successfully.")
        log.info(f"Number of scenes: {len(nusc.scene)}")
    except Exception as e:
        log.error(f"Error loading NuScenes: {e}", exc_info=True)
        nusc = None
else:
    log.warning(f"NuScenes dataroot not found or not specified: {NU_DATAROOT}")

2025-05-07 17:02:24,182 - NuScenesNotebook - INFO - Initializing DynObjFilter with config: /home/drugge/Unsupervised-Moving-Point-Detection/test_data/configs/test_full_config_low_log.yaml
2025-05-07 17:02:24,185 - NuScenesNotebook - INFO - DynObjFilter initialized successfully.
2025-05-07 17:02:24,186 - NuScenesNotebook - INFO - Filter initial map count: 0
2025-05-07 17:02:24,187 - NuScenesNotebook - INFO - Loading NuScenes (Version: v1.0-mini, Dataroot: /datasets/nuscenes)...


[2025-05-07 17:02:24.185] [info] [Logging Setup] Logging initialized. Console base level: 'info'. Default per-sequence file level: 'info'. Log directory: '/tmp/mdet_logs'
[2025-05-07 17:02:24.185] [Filter_Core] [info] DynObjFilter constructed. History length: 10
[2025-05-07 17:02:24.185] [Filter_Core] [info]   RingBuffer capacity: 10
[2025-05-07 17:02:24.185] [Filter_Core] [info] DynObjFilter initialization complete.


2025-05-07 17:02:24,858 - NuScenesNotebook - INFO - NuScenes loaded successfully.
2025-05-07 17:02:24,859 - NuScenesNotebook - INFO - Number of scenes: 10


In [3]:

sample_tokens_to_process = []
scene_name_for_run = "N/A"

if nusc and len(nusc.scene) > NU_SCENE_INDEX_TO_LOAD:
    scene_info = nusc.scene[NU_SCENE_INDEX_TO_LOAD]
    scene_name_for_run = scene_info['name']
    log.info(f"Selected Scene: {scene_name_for_run} (Index {NU_SCENE_INDEX_TO_LOAD})")

    current_sample_token = scene_info['first_sample_token']
    for _ in range(NU_NUM_SAMPLES_TO_PROCESS):
        if not current_sample_token:
            log.warning("Reached end of scene before collecting desired number of samples.")
            break
        sample_tokens_to_process.append(current_sample_token)
        sample_record = nusc.get('sample', current_sample_token)
        current_sample_token = sample_record['next']
    log.info(f"Collected {len(sample_tokens_to_process)} sample tokens for processing.")
elif nusc:
    log.error(f"Scene index {NU_SCENE_INDEX_TO_LOAD} is out of range. Max index: {len(nusc.scene)-1}")
else:
    log.error("NuScenes not loaded. Cannot select samples.")


2025-05-07 17:02:24,871 - NuScenesNotebook - INFO - Selected Scene: scene-0061 (Index 0)
2025-05-07 17:02:24,872 - NuScenesNotebook - INFO - Collected 10 sample tokens for processing.


In [ ]:
# Cell 5: Interactive C++ Utility Function Calls (Corrected)

if filt and mdet and nh and nusc and 'points_sensor_frame' in locals() and 'rotation_s2w' in locals() and 'position_s2w' in locals():
    log.info("\n--- Interactive C++ Utility Testing (with Real Data) ---")

    # Use data from the LAST scan processed in Cell 4
    # points_sensor_frame: Nx4 points in sensor frame (from last iteration of Cell 4)
    # rotation_s2w, position_s2w: Sensor-to-World transform (from last iteration of Cell 4)

    if points_sensor_frame.shape[0] > 0:
        test_point_idx_in_scan = 0  # Test with the first point of the last scan
        
        # 1. Get the point in sensor frame (X, Y, Z only for transformation)
        p_sensor_xyz = points_sensor_frame[test_point_idx_in_scan, :3]

        # 2. Transform this point to global coordinates
        # p_global = R_s2w * p_sensor + T_s2w
        p_global_xyz = rotation_s2w @ p_sensor_xyz + position_s2w
        global_point_coords_for_binding = p_global_xyz.reshape(3, 1).astype(np.float64) # Ensure 3x1 and float64

        # 3. The depth_index is the original index of the point in its scan
        depth_index_for_binding = test_point_idx_in_scan # Or if points_sensor_frame was filtered, use its original index if available

        # 4. Calculate World-to-Sensor transformation
        # R_w2s = R_s2w.T
        # T_w2s = -R_s2w.T * T_s2w
        rotation_w2s_for_binding = rotation_s2w.T.astype(np.float64)
        translation_w2s_for_binding = (-rotation_s2w.T @ position_s2w).reshape(3, 1).astype(np.float64)

        log.info(f"Testing spherical_projection with point from last scan (original index: {test_point_idx_in_scan})")
        log.info(f"  Global Coords: {global_point_coords_for_binding.flatten().tolist()}")
        log.info(f"  Rotation (W2S): \n{rotation_w2s_for_binding}")
        log.info(f"  Translation (W2S): {translation_w2s_for_binding.flatten().tolist()}")
        log.info(f"  Config Path: {CONFIG_PATH_STR}")

        try:
            sph_coords_m_tuple = mdet.spherical_projection(
                global_point_coords=global_point_coords_for_binding,
                original_idx_for_point=test_point_idx_in_scan, # <--- UPDATED ARGUMENT NAME
                rotation_world_to_sensor=rotation_w2s_for_binding,
                translation_world_to_sensor=translation_w2s_for_binding,
                config_path=CONFIG_PATH_STR
            )
            # Unpack the tuple
            sph_coords_m_numpy_array = sph_coords_m_tuple[0] # This is the py::array_t<float>
            H_ind = sph_coords_m_tuple[1]
            V_ind = sph_coords_m_tuple[2]
            GridPos = sph_coords_m_tuple[3]

            # Convert the numpy array for sph_coords_m to a simple list or ensure it's handled correctly downstream
            # For direct use, it's fine. If you were expecting a list:
            sph_coords_m = list(sph_coords_m_numpy_array) # sph_coords_m is now [az, el, depth]

            log.info(f"C++ spherical_projection output:")
            log.info(f"  Spherical Coords (Az, El, Depth): {sph_coords_m}")
            log.info(f"  H_ind: {H_ind}, V_ind: {V_ind}, GridPos: {GridPos}")


        except Exception as e:
            log.error(f"Error calling spherical_projection: {e}", exc_info=True)
    else:
        log.warning("Last processed scan had no points, cannot test spherical_projection with its data.")
else:
    log.warning("Filter, mdet, helpers, NuScenes, or data from previous cells not available for interactive C++ utility tests.")

2025-05-07 13:34:54,144 - NuScenesNotebook - INFO - 
--- Interactive C++ Utility Testing (with Real Data) ---
2025-05-07 13:34:54,146 - NuScenesNotebook - INFO - Testing spherical_projection with point from last scan (original index: 0)
2025-05-07 13:34:54,147 - NuScenesNotebook - INFO -   Global Coords: [402.71701374476726, 1145.7135467863761, -0.08218456925105411]
2025-05-07 13:34:54,152 - NuScenesNotebook - INFO -   sph_coords_m raw output: [-3.0503809e+00 -1.0704099e-01  3.5090302e+02]
2025-05-07 13:34:54,153 - NuScenesNotebook - INFO -   sph_coords_m.shape: (3,)
2025-05-07 13:34:54,154 - NuScenesNotebook - INFO -   sph_coords_m.ndim: 1
2025-05-07 13:34:54,155 - NuScenesNotebook - INFO -   spherical_projection Result: Spherical(rad)=(Az: -3.050, El: -0.107), Depth=350.90m, H_ind=13, V_ind=66, GridPos=5903


In [ ]:
# Ensure these variables are from the same context as the spherical_projection call
# global_point_coords_for_binding (3x1 np.float64)
# rotation_w2s_for_binding (3x3 np.float64)
# translation_w2s_for_binding (3x1 np.float64)

# Calculate point coordinates in sensor frame: p_sensor = R_w2s * p_global + T_w2s
point_in_sensor_frame_xyz = rotation_w2s_for_binding @ global_point_coords_for_binding + translation_w2s_for_binding
# point_in_sensor_frame_xyz should be a 3x1 array. Let's flatten for easier use.
x_s = point_in_sensor_frame_xyz[0, 0]
y_s = point_in_sensor_frame_xyz[1, 0]
z_s = point_in_sensor_frame_xyz[2, 0]

log.info(f"Point in Sensor Frame (calculated in Python): X={x_s:.3f}, Y={y_s:.3f}, Z={z_s:.3f}")

# Manual calculation of spherical coordinates
manual_depth = np.sqrt(x_s**2 + y_s**2 + z_s**2)
# Azimuth: atan2(y,x). Ensure your C++ uses the same convention (e.g. x-forward, y-left, z-up for sensor)
# Common convention: x forward, y left, z up. Azimuth is angle in x-y plane from x-axis.
manual_azimuth_rad = np.arctan2(y_s, x_s)
# Elevation: angle with xy-plane.
manual_elevation_rad = np.arctan2(z_s, np.sqrt(x_s**2 + y_s**2)) # More robust: np.arcsin(z_s / manual_depth) if manual_depth > 0

log.info(f"Manually Calculated Spherical Coords (Sensor Frame):")
log.info(f"  Depth: {manual_depth:.2f} m")
log.info(f"  Azimuth: {manual_azimuth_rad:.3f} rad ({np.rad2deg(manual_azimuth_rad):.1f} deg)")
log.info(f"  Elevation: {manual_elevation_rad:.3f} rad ({np.rad2deg(manual_elevation_rad):.1f} deg)")

log.info(f"C++ Output Spherical Coords:")
log.info(f"  Depth: {sph_coords_m[2]:.2f} m")
log.info(f"  Azimuth: {sph_coords_m[0]:.3f} rad ({np.rad2deg(sph_coords_m[0]):.1f} deg)")
log.info(f"  Elevation: {sph_coords_m[1]:.3f} rad ({np.rad2deg(sph_coords_m[1]):.1f} deg)")

2025-05-07 13:34:58,677 - NuScenesNotebook - INFO - Point in Sensor Frame (calculated in Python): X=-3.160, Y=-0.306, Z=-1.885
2025-05-07 13:34:58,680 - NuScenesNotebook - INFO - Manually Calculated Spherical Coords (Sensor Frame):
2025-05-07 13:34:58,681 - NuScenesNotebook - INFO -   Depth: 3.69 m
2025-05-07 13:34:58,682 - NuScenesNotebook - INFO -   Azimuth: -3.045 rad (-174.5 deg)
2025-05-07 13:34:58,684 - NuScenesNotebook - INFO -   Elevation: -0.536 rad (-30.7 deg)
2025-05-07 13:34:58,685 - NuScenesNotebook - INFO - C++ Output Spherical Coords:
2025-05-07 13:34:58,686 - NuScenesNotebook - INFO -   Depth: 350.90 m
2025-05-07 13:34:58,687 - NuScenesNotebook - INFO -   Azimuth: -3.050 rad (-174.8 deg)
2025-05-07 13:34:58,688 - NuScenesNotebook - INFO -   Elevation: -0.107 rad (-6.1 deg)


In [ ]:
# Load config for spherical projection parameters
sph_proj_config = None
if CONFIG_PATH_STR:
    try:
        with open(CONFIG_PATH_STR, 'r') as f:
            full_cfg = yaml.safe_load(f)
        if full_cfg and 'dyn_obj' in full_cfg and 'spherical_projection' in full_cfg['dyn_obj']:
            sph_proj_config = full_cfg['dyn_obj']['spherical_projection']
            log.info(f"Loaded spherical projection config: {sph_proj_config}")
        else:
            log.error("Spherical projection config not found in YAML.")
    except Exception as e:
        log.error(f"Error loading spherical projection config: {e}")

if sph_proj_config:
    num_hor_cells = sph_proj_config.get('num_horizontal_cells', 1024) # Default from your C++ if not in YAML
    num_ver_cells = sph_proj_config.get('num_vertical_cells', 90)
    hor_fov_deg = sph_proj_config.get('horizontal_fov_deg', 360.0)
    min_ver_angle_deg = sph_proj_config.get('min_vertical_angle_deg', -25.0)
    max_ver_angle_deg = sph_proj_config.get('max_vertical_angle_deg', 2.0) # From your YAML

    # Derived parameters
    total_ver_fov_deg = max_ver_angle_deg - min_ver_angle_deg
    hor_res_rad = np.deg2rad(hor_fov_deg) / num_hor_cells
    ver_res_rad = np.deg2rad(total_ver_fov_deg) / num_ver_cells
    
    # Azimuth start for index 0 (common: -PI or 0)
    # If 360 FOV, often mapping [-PI, PI) to [0, NUM_HOR_CELLS-1]
    # Let's assume azimuth_map_offset_rad makes the C++ azimuth fall into [0, 2*PI] before indexing
    # Or, if C++ azimuth is in [-PI, PI), map_azimuth = cpp_azimuth + PI
    azimuth_start_rad = -np.pi # Assuming range starts at -180 deg for index calculation

    # Elevation start for index 0
    elevation_start_rad = np.deg2rad(min_ver_angle_deg)

    # Use the azimuth and elevation from the C++ output for this check
    cpp_azimuth_rad = sph_coords_m[0]
    cpp_elevation_rad = sph_coords_m[1]

    # Manual Horizontal Index Calculation
    # Adjust cpp_azimuth_rad if its range doesn't match [azimuth_start_rad, azimuth_start_rad + hor_fov_rad)
    # e.g., if cpp_azimuth_rad is [-PI, PI] and hor_fov is 2*PI starting at -PI
    # effective_azimuth = cpp_azimuth_rad - azimuth_start_rad # Should be in [0, hor_fov_rad)
    # A common way for 360 FOV:
    # norm_azimuth_rad = cpp_azimuth_rad + np.pi # Convert from [-pi, pi] to [0, 2*pi]
    # manual_h_ind = np.floor(norm_azimuth_rad / hor_res_rad) % num_hor_cells
    
    # Simpler: if C++ azimuth is -174.7 deg, and hor_fov is 360, starting at -180.
    # Azimuth relative to start of FOV:
    az_rel_to_fov_start_rad = cpp_azimuth_rad - azimuth_start_rad # (-3.050 - (-3.1415)) = 0.091 rad
    manual_h_ind = int(np.floor(az_rel_to_fov_start_rad / hor_res_rad))
    # Ensure it's within bounds (e.g. for angles slightly outside due to precision)
    manual_h_ind = max(0, min(manual_h_ind, num_hor_cells - 1))


    # Manual Vertical Index Calculation
    # Elevation relative to start of FOV:
    el_rel_to_fov_start_rad = cpp_elevation_rad - elevation_start_rad
    manual_v_ind = int(np.floor(el_rel_to_fov_start_rad / ver_res_rad))
    # Ensure it's within bounds
    manual_v_ind = max(0, min(manual_v_ind, num_ver_cells - 1))


    log.info(f"Manually Calculated Grid Indices (based on C++ angles & config):")
    log.info(f"  H_ind: {manual_h_ind} (C++ H_ind: {H_ind})")
    log.info(f"  V_ind: {manual_v_ind} (C++ V_ind: {V_ind})")

    # Manual GridPos Calculation (common: row-major)
    # This depends on how your C++ code calculates it.
    # Option 1: Row-major (V_ind varies slowest)
    manual_grid_pos_row_major = manual_v_ind * num_hor_cells + manual_h_ind
    # Option 2: Column-major (H_ind varies slowest)
    manual_grid_pos_col_major = manual_h_ind * num_ver_cells + manual_v_ind
    
    log.info(f"Manually Calculated GridPos:")
    log.info(f"  Row-Major: {manual_grid_pos_row_major}")
    log.info(f"  Col-Major: {manual_grid_pos_col_major}")
    log.info(f"  C++ GridPos: {GridPos}")

2025-05-07 13:07:53,296 - NuScenesNotebook - ERROR - Spherical projection config not found in YAML.
